In [1]:
import os, sys, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, os.getcwd())
try:
    from 集成公共函数 import *
except ImportError as e:
    print('导入 集成公共函数.py 失败:', e)
    raise

In [ ]:
ALPHA = 0.10          #90%预测区间
SEED  = 0

In [3]:
ART = load_or_build_models()     
ENSEMBLES     = ART['ENSEMBLES']
TUNED         = ART['TUNED']
W_raw         = ART['W_raw']
WEIGHTS_TABLE = ART['WEIGHTS_TABLE']
CV_SCORES     = ART['CV_SCORES']
BEST_PARAMS   = ART['BEST_PARAMS']
TUNE_META     = ART['TUNE_META']


print('已加载目标:', list(ENSEMBLES.keys()))
print('各目标非零权重成员数:', {(TAG_CN[k], int((W_raw[k] > 0.001).sum())) for k in ENSEMBLES})

[加载] 已从存档载入集成模型: d:\Code_File\Python_Data_AI\Project_CFPA_Teacher\Data\集成建模\集成模型存档.joblib (version=3)
已加载目标: ['TS', 'TM', 'FS', 'FM', 'IS']
各目标非零权重成员数: {('拉伸强度', 4), ('冲击强度', 3), ('弯曲强度', 3), ('弯曲模量', 5), ('弹性模量', 4)}


In [4]:
NESTED = ART.get('NESTED', None)
if NESTED is None:
    print('存档中没有 NESTED 计算')
    NESTED = {}
    for tag in SHEETS.values():
        t0 = time.time()
        NESTED[tag] = nested_oof_v2(tag, tuned=TUNED[tag], nested_tune=NESTED_TUNE,
                                    n_outer=N_OUTER, n_inner=N_INNER, verbose=False)
        print(f"  [{tag}] {TAG_CN[tag]} 嵌套OOF完成({time.time()-t0:.0f}s) "
              f"OOF-R2={r2_score(NESTED[tag]['y'], NESTED[tag]['oof']):.3f}")
else:
    print('[复用] 已从模块一存档载入诚实嵌套OOF结果:',
          {t: (NESTED[t]['group_mode'], bool(NESTED[t]['nested_tune'])) for t in SHEETS.values()})
DIAG = NESTED
for tag in SHEETS.values():
    print(f"  [{tag}] {TAG_CN[tag]} OOF-R2={r2_score(DIAG[tag]['y'], DIAG[tag]['oof']):.3f}")

[复用] 已从模块一存档载入诚实嵌套OOF结果: {'TS': ('combo', True), 'TM': ('combo', True), 'FS': ('combo', True), 'FM': ('combo', True), 'IS': ('combo', True)}
  [TS] 拉伸强度 OOF-R2=0.341
  [TM] 弹性模量 OOF-R2=0.280
  [FS] 弯曲强度 OOF-R2=0.779
  [FM] 弯曲模量 OOF-R2=0.571
  [IS] 冲击强度 OOF-R2=0.332


In [5]:
#用嵌套OOF残差做共形校准
def cond_agg(v, cond):
    return pd.DataFrame({'v': np.asarray(v, float), 'cond': np.asarray(cond)}).groupby('cond')['v'].mean().values

IE = {}
for tag in SHEETS.values():
    cm = load_condition_means(tag)
    yc = cm['y'].values.astype(float)
    pc = cond_agg(DIAG[tag]['oof'], DIAG[tag]['d']['cond'].values)
    ie = IntervalEnsemble(ENSEMBLES[tag], cm[FEATS].values.astype(float),
                          yc, cm['cond'].values)
    ie.calibrate_conformal(pc, yc, alpha=ALPHA)
    IE[tag] = ie
    print(f"  [{tag}] {TAG_CN[tag]} 共形半宽 q̂={ie.qhat_:.2f}")

  [TS] 拉伸强度 共形半宽 q̂=11.05
  [TM] 弹性模量 共形半宽 q̂=1976.33
  [FS] 弯曲强度 共形半宽 q̂=15.32
  [FM] 弯曲模量 共形半宽 q̂=799.68
  [IS] 冲击强度 共形半宽 q̂=4.14


In [6]:
#各目标共形区间(训练OOF 外部验证)
INTERVALS = {}
for tag in SHEETS.values():
    pc = cond_agg(DIAG[tag]['oof'], DIAG[tag]['d']['cond'].values)
    Xe = load_external()[tag][FEATS].values.astype(float)
    q = IE[tag].qhat_
    lo_tr, hi_tr, mu_tr = pc - q, pc + q, pc
    lo_ex, hi_ex, mu_ex = IE[tag].conformal_interval(Xe)
    INTERVALS[tag] = dict(tr=(lo_tr, hi_tr, mu_tr),
                          ex=(lo_ex, hi_ex, mu_ex))

In [7]:
# 指标汇总表
rows = []
for tag in SHEETS.values():
    y_tr = load_condition_means(tag)['y'].values.astype(float)
    ye   = load_external()[tag]['y_mean'].values.astype(float)
    lo, hi, mu = INTERVALS[tag]['tr']
    tr = interval_metrics(y_tr, lo, hi)
    lo, hi, mu = INTERVALS[tag]['ex']
    ex = interval_metrics(ye, lo, hi)
    rows.append(dict(目标=TAG_CN[tag],
                     训练_PICP=round(tr['PICP'],3), 训练_MPIW=round(tr['MPIW'],2), 训练_PINAW=round(tr['PINAW'],3),
                     外部_PICP=round(ex['PICP'],3), 外部_MPIW=round(ex['MPIW'],2), 外部_PINAW=round(ex['PINAW'],3)))
metric_table = pd.DataFrame(rows)
metric_table

,目标,训练_PICP,训练_MPIW,训练_PINAW,外部_PICP,外部_MPIW,外部_PINAW
0,拉伸强度,0.919,22.11,0.499,1.00,22.11,1.154
1,弹性模量,0.919,3952.66,0.483,1.00,3952.66,1.296
2,弯曲强度,0.919,30.63,0.372,1.00,30.63,0.728
3,弯曲模量,0.919,1599.37,0.410,0.75,1599.37,0.520
4,冲击强度,0.919,8.28,0.514,1.00,8.28,1.020


In [8]:
#测试  对新工艺参数直接输出点预测 + 区间
x = np.array([[310, 90, 30, 0.1, 1]])      # 喷嘴温度310°C, 热床90°C, 速度30, 层厚0.1mm, 方向1
print('特征顺序:', FEATS)


for tag in SHEETS.values():
    cm = load_condition_means(tag)
    yc = cm['y'].values.astype(float)
    pc = cond_agg(DIAG[tag]['oof'], DIAG[tag]['d']['cond'].values)
    IE[tag].calibrate_conformal(pc, yc, alpha=ALPHA)
for tag in SHEETS.values():
    lo, hi, mu = IE[tag].conformal_interval(x)
    print(f"{TAG_CN[tag]:6s} ({tag}): 点预测={mu[0]:7.2f} | {(1-ALPHA)*100:.0f}%区间=[{lo[0]:7.2f}, {hi[0]:7.2f}] 宽={hi[0]-lo[0]:.2f}")

特征顺序: ['T_noz', 'T_bed', 'speed', 'layer', 'direction']
拉伸强度   (TS): 点预测= 130.14 | 90%区间=[ 119.09,  141.19] 宽=22.11
弹性模量   (TM): 点预测=14510.25 | 90%区间=[12533.92, 16486.58] 宽=3952.66
弯曲强度   (FS): 点预测= 165.65 | 90%区间=[ 150.34,  180.97] 宽=30.63
弯曲模量   (FM): 点预测=9097.75 | 90%区间=[8298.06, 9897.43] 宽=1599.37
冲击强度   (IS): 点预测=  26.81 | 90%区间=[  22.67,   30.96] 宽=8.28
